### Load dataset

In [1]:
import torch
import plotly.express as px

from src import configs as cfg
from src import models, dataset

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
path = "checkpoints/swin_MiM/wise-morning-649/swin_ssl_epoch_1750.pt"
chkpt = torch.load(path, weights_only=False)
train_cfg = chkpt["train_cfg"]
if "mask_rec_loss_weight" in train_cfg:
    del train_cfg["mask_rec_loss_weight"]
train_cfg = cfg.TrainingConfig(**train_cfg)
model_cfg = cfg.ModelConfig(**chkpt["model_cfg"])
model = models.mk_model_from_cfg(model_cfg)
model.load_state_dict(chkpt["model"])

<All keys matched successfully>

In [3]:
loaders = dataset.mk_ssl_loaders(train_cfg)
batch_dict = next(iter(loaders["train"]))
batch_dict = dataset.preprocess_batch(batch_dict)
with torch.no_grad(), torch.autocast(cfg.DEVICE.type, torch.bfloat16):
    model_output = model(batch_dict)

In [4]:
model_output.keys()

dict_keys(['loss', 'reconstruction', 'hidden_states', 'attentions', 'reshaped_hidden_states', 'x_hat'])

In [10]:
img = model_output["x_hat"].detach().cpu().float().numpy()
img.shape
px.imshow(
    img[10:20, 0],
    facet_col=0,
    color_continuous_scale="rainbow",
    facet_col_wrap=5,
)

In [11]:
batch_dict = next(iter(loaders["valid"]))
batch_dict = dataset.preprocess_batch(batch_dict)
with torch.no_grad(), torch.autocast(cfg.DEVICE.type, torch.bfloat16):
    model_output = model(batch_dict)

img = model_output["x_hat"].detach().cpu().float().numpy()
img.shape
px.imshow(
    img[10:20, 0],
    facet_col=0,
    color_continuous_scale="rainbow",
    facet_col_wrap=5,
)